## Sequential Workflow with MAF and Microsoft Foundry

![sequential-workflow](./Assets/sequential-workflow.png)

In [ ]:
%pip install -U agent-framework-core agent-framework-foundry azure-ai-projects azure-identity python-dotenv ipython

### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv


load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://udemo-103-resource.services.ai.azure.com/api/projects/udemo-103
Model:  gpt-5.2


### Defining the Function to Create a Chat Agent

In [ ]:
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential
from agent_framework import Agent

# Cell 2: Define async workflow
async def create_agent(agent_name: str,
                       agent_instructions: str) -> Agent:
    
    # Create async Azure credential
    credential = DefaultAzureCredential()

    # creating the Foundry Project Client
    chat_client = FoundryChatClient(
        project_endpoint=project_endpoint,
        model=model,
        credential=credential,
    )
    
    agent = Agent(
            name=agent_name,
            instructions=agent_instructions,
            client=chat_client,
        )

    print("{} Agent created successfully!".format(agent_name))
    return agent

### Creating the Researcher Agent

In [4]:
researcher_agent = await create_agent(
    agent_name="Researcher-Agent",
    agent_instructions="You are a knowledgeable researcher. Your task is to gather information and provide insights on a given topic. "
                       "You should use reliable sources and present the information in a clear and concise manner."
    )

Researcher-Agent Agent created successfully!


### Creating the Writer Agent

In [5]:
writer_agent = await create_agent(
    agent_name="Writer-Agent",
    agent_instructions="You are a creative writer. Your task is to write an essay on a given topic. "
                       "You should focus on clarity, coherence, and engaging storytelling."
    )
     

Writer-Agent Agent created successfully!


### Creating the First Executor to run the Researcher Agent

In [19]:
from agent_framework import  WorkflowContext, executor

@executor(id="run_researcher_agent")
async def run_researcher_agent(
    query: str,
    ctx: WorkflowContext[str],
) -> None:
    response = await researcher_agent.run(query)
    await ctx.send_message(str(response.text))


### Creating the Second Executor to run the Writer Agent

In [20]:
@executor(id="run_writer_agent")
async def run_writer_agent(
    research_data: str,
    ctx: WorkflowContext[str],
) -> None:
    response = await writer_agent.run(research_data)
    await ctx.yield_output(str(response.text))

### Creating the Sequential Workflow

In [21]:
from agent_framework import WorkflowBuilder, WorkflowViz

workflow = (
    WorkflowBuilder(start_executor=run_researcher_agent)
    .add_edge(run_researcher_agent, run_writer_agent)
    .build()
)

viz = WorkflowViz(workflow)

### Generating Mermaid Diagram for Visualization

In [22]:
from IPython.display import display, Markdown

mermaid_content = viz.to_mermaid()
display(Markdown(f"```mermaid\n{mermaid_content}\n```"))

```mermaid
flowchart TD
  run_researcher_agent["run_researcher_agent (Start)"];
  run_writer_agent["run_writer_agent"];
  run_researcher_agent --> run_writer_agent;
```

### Running the Workflow and Streaming Events

In [ ]:
async def main():
    prompt = "Write an essay about the impact of AI on society."

    result = await workflow.run(prompt)

    print("Workflow completed with result:")
    outputs = result.get_outputs()

    for output in outputs:
        print(output)

await main()

Workflow completed with result:
Artificial intelligence arrived the way electricity once did: first as a laboratory marvel, then as a quiet utility, and finally as an invisible force reorganizing daily life. At the beginning, you could point to it—a chess-playing program, a research paper, a demo that felt like a trick. Now it hums behind the scenes of hiring portals and hospital scanners, customer service chat windows and logistics routes, photo apps and fraud alerts. AI’s social impact is not a single story of progress or peril. It is a story of *amplified capability*—prediction, pattern recognition, content generation, decision support—colliding with old human realities: unequal power, imperfect institutions, and the persistent temptation to trade accountability for efficiency.

### The new engine room of the economy

In offices and supply chains, AI often behaves less like a robot replacing a person and more like a new “middle layer” between intention and execution. A manager asks 

In [24]:
# Cleanup: close agents

await researcher_agent.close()
await writer_agent.close()

print("Cleanup completed: agents are closed.")

Cleanup completed: agents are closed.
